# 1 字符串输出解析器 StrOutputParser

In [1]:
# 获取大模型
import os
from langchain_openai import ChatOpenAI
import dotenv

dotenv.load_dotenv(override=True)
prefix = "DAIPAI_"
model_name = os.getenv(prefix + "MODEL")
model_base_url = os.getenv(prefix + "BASE_URL")
model_api_key = os.getenv(prefix + "API_KEY")

print(f"Model Name: {model_name}")


chat_llm = ChatOpenAI(model=model_name or "gpt-3.5-turbo", base_url=model_base_url, api_key=model_api_key) # type: ignore

Model Name: daipai/qwen/qwen3-coder-480b-a35b-instruct


In [2]:
# 调用content 输出字符串
res = chat_llm.invoke("Hello, world!")
print(res.content)

Hello! It looks like you're saying hello to the world. How can I help you today? Are you looking for information, trying to learn something new,


In [3]:
# 使用string output parser
from langchain_core.output_parsers import StrOutputParser


parser = StrOutputParser()
# With streaming - use transform() to process a stream
stream = chat_llm.stream("给我讲一个笑话")
for chunk in parser.transform(stream):
    print(chunk, end="", flush=True)

好的，给你讲一个轻松的笑话：

一天，小明去买水果。

小明问老板："这苹果怎么卖？"
老板说："5块钱

In [4]:
parser = StrOutputParser()
str = parser.invoke(res)
print(str)
print(type(str))

Hello! It looks like you're saying hello to the world. How can I help you today? Are you looking for information, trying to learn something new,
<class 'langchain_core.messages.base.TextAccessor'>


# 2 json输出解析器 JsonOutputParser

In [5]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system","你是一个靠谱的{role}，请根据用户的输入进行回答。你输出的格式需要是标准可解析的json字符串"),
        ("human","{user_input}"),
    ]
)
prompt = prompt.format_messages(role="助手", user_input="请你介绍一下自己")

res = chat_llm.invoke(prompt)
print(res.content)

{
  "response": "我是通义千问，是阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我能够回答问题、创作


In [6]:
# import json

from langchain_core.output_parsers import JsonOutputParser


parser = JsonOutputParser()
json_result = parser.invoke(res)
print(json_result)
print(type(json_result))

{'response': '我是通义千问，是阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我能够回答问题、创作'}
<class 'dict'>


In [7]:
parser.get_format_instructions()

'Return a JSON object.'

In [8]:
from langchain_core.prompts import PromptTemplate


prompt_template = PromptTemplate.from_template(
    template="回答以下问题：{question}，请以{format}格式输出答案",
    partial_variables={"format": parser.get_format_instructions()}
)

prompt = prompt_template.invoke({"question": "请介绍一下西工大"})
print(prompt)

res = chat_llm.invoke(prompt)
json_result = parser.invoke(res)
print(json_result)

text='回答以下问题：请介绍一下西工大，请以Return a JSON object.格式输出答案'
{'学校名称': '西北工业大学', '英文名称': 'Northwestern Polytechnical University', '简称': '西工大'}


In [9]:
chain = prompt_template | chat_llm | parser
res = chain.invoke({"question": "请介绍一下西工大"})
print(res)

{'学校名称': '西北工业大学', '英文名称': 'Northwestern Polytechnical University', '简称': '西工大'}


# 3 XML输出解析器 XmlOutputParser

In [ ]:
from langchain_core.output_parsers import XMLOutputParser


parser = XMLOutputParser()
prompt_template = PromptTemplate.from_template(
    template="""回答以下问题：{question}
    
请严格按照以下 XML 格式输出答案：
{format}

注意：
1. 只输出 XML 内容，不要包含 ```xml 或 ``` 等 markdown 标记
2. 确保 XML 完整且格式正确
3. 不要输出任何额外说明文字""", 
    partial_variables={"format": parser.get_format_instructions()}
)
res = chat_llm.invoke(prompt_template.invoke({"question": "请介绍一下西工大"}))
print(res.content)
# xml_result = parser.invoke(res)
# print(xml_result)

<university>
    <name>西北工业大学</name>
    <english_name>Northwestern Polytechnical University</english_name>
    <location>陕西省


# 列表解析器 CommaSeparatedListOutputParser

In [1]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser


parser = CommaSeparatedListOutputParser()
print(parser.get_format_instructions())

message = "大象,猩猩,狮子"
result = parser.invoke(message)
print(result)


Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['大象', '猩猩', '狮子']


# 日期解析器 DatetimeOutputParser

In [4]:
from langchain_classic.output_parsers import DatetimeOutputParser

a = DatetimeOutputParser()
print(a.get_format_instructions())

Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.

Examples: 2023-07-04T14:30:00.000000Z, 1999-12-31T23:59:59.999999Z, 2025-01-01T00:00:00.000000Z

Return ONLY this string, no other words!
